## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab with a GPU.

**Colab workflow:**
- Mount Google Drive to persist trained checkpoints and results.
- Download source, data, and the baseline checkpoint from GitHub.
- Physics features (RSA, disorder, φ) are stored as compact JSON.gz files (≤ 14 MB each) committed to the repo — no manual upload needed.
- Download a fresh ESM-2 snapshot from Hugging Face.

**GitHub artifacts required:**
- `src/xallergen/baseline_notebook_utils.py`
- `src/xallergen/physics_head/` (all 5 Python files)
- `data/deepalgpro_train_cleaned.csv`, `data/deepalgpro_test_cleaned.csv`, `data/positives_splitB.csv`
- `data/rsa/deepalgpro_train_rsa.json.gz`, `deepalgpro_train_disorder.json.gz`, `deepalgpro_train_phi.json.gz`
- `data/rsa/deepalgpro_test_rsa.json.gz`, `deepalgpro_test_disorder.json.gz`, `deepalgpro_test_phi.json.gz`
- `data/ss3/iedb_positive_sequences_predictions.csv`
- `results/rsa_regularization/lambda_0/baseline_frozen_esm2.pt`

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    DRIVE_ROOT = None
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

SRC_DIR      = RUNTIME_ROOT / 'src'
PACKAGE_DIR  = SRC_DIR / 'xallergen'
PHYS_DIR     = PACKAGE_DIR / 'physics_head'
DATA_DIR     = RUNTIME_ROOT / 'data'
RSA_DIR      = DATA_DIR / 'rsa'
SS3_DIR      = DATA_DIR / 'ss3'
CKPT_BASELINE = RUNTIME_ROOT / 'results' / 'rsa_regularization' / 'lambda_0' / 'baseline_frozen_esm2.pt'
HF_DIR       = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'

# Output: Drive on Colab, local results/ otherwise
if IS_COLAB:
    RESULTS_DIR = DRIVE_ROOT / 'results' / 'physics_head'
else:
    RESULTS_DIR = RUNTIME_ROOT / 'results' / 'physics_head'

for path in [SRC_DIR, PACKAGE_DIR, PHYS_DIR, DATA_DIR, RSA_DIR, SS3_DIR,
             CKPT_BASELINE.parent, HF_DIR, RESULTS_DIR, RESULTS_DIR / 'checkpoints']:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

    download_jobs = [
        # xallergen package
        (f'{RAW}/src/xallergen/__init__.py',                        PACKAGE_DIR / '__init__.py'),
        (f'{RAW}/src/xallergen/baseline_notebook_utils.py',         PACKAGE_DIR / 'baseline_notebook_utils.py'),
        # physics_head subpackage
        (f'{RAW}/src/xallergen/physics_head/__init__.py',           PHYS_DIR / '__init__.py'),
        (f'{RAW}/src/xallergen/physics_head/features.py',           PHYS_DIR / 'features.py'),
        (f'{RAW}/src/xallergen/physics_head/model.py',              PHYS_DIR / 'model.py'),
        (f'{RAW}/src/xallergen/physics_head/train.py',              PHYS_DIR / 'train.py'),
        (f'{RAW}/src/xallergen/physics_head/evaluate.py',           PHYS_DIR / 'evaluate.py'),
        # sequence data
        (f'{RAW}/data/deepalgpro_train_cleaned.csv',                DATA_DIR / 'deepalgpro_train_cleaned.csv'),
        (f'{RAW}/data/deepalgpro_test_cleaned.csv',                 DATA_DIR / 'deepalgpro_test_cleaned.csv'),
        (f'{RAW}/data/positives_splitB.csv',                        DATA_DIR / 'positives_splitB.csv'),
        # physics features — compact JSON.gz (RSA, disorder, phi)
        (f'{RAW}/data/rsa/deepalgpro_train_rsa.json.gz',            RSA_DIR / 'deepalgpro_train_rsa.json.gz'),
        (f'{RAW}/data/rsa/deepalgpro_train_disorder.json.gz',       RSA_DIR / 'deepalgpro_train_disorder.json.gz'),
        (f'{RAW}/data/rsa/deepalgpro_train_phi.json.gz',            RSA_DIR / 'deepalgpro_train_phi.json.gz'),
        (f'{RAW}/data/rsa/deepalgpro_test_rsa.json.gz',             RSA_DIR / 'deepalgpro_test_rsa.json.gz'),
        (f'{RAW}/data/rsa/deepalgpro_test_disorder.json.gz',        RSA_DIR / 'deepalgpro_test_disorder.json.gz'),
        (f'{RAW}/data/rsa/deepalgpro_test_phi.json.gz',             RSA_DIR / 'deepalgpro_test_phi.json.gz'),
        # IEDB predictions for splitB epitope alignment
        (f'{RAW}/data/ss3/iedb_positive_sequences_predictions.csv', SS3_DIR / 'iedb_positive_sequences_predictions.csv'),
        # baseline checkpoint
        (f'{RAW}/results/rsa_regularization/lambda_0/baseline_frozen_esm2.pt', CKPT_BASELINE),
    ]

    failed = []
    for url, dst in download_jobs:
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as exc:
            failed.append((url, str(exc)))
    if failed:
        raise RuntimeError('Failed downloads:\n' + '\n'.join(f'  {u} -> {e}' for u, e in failed))

    fresh_esm2 = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_esm2)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Validate required files exist before proceeding
required = [
    DATA_DIR / 'deepalgpro_train_cleaned.csv',
    DATA_DIR / 'deepalgpro_test_cleaned.csv',
    DATA_DIR / 'positives_splitB.csv',
    RSA_DIR  / 'deepalgpro_train_rsa.json.gz',
    RSA_DIR  / 'deepalgpro_train_disorder.json.gz',
    RSA_DIR  / 'deepalgpro_train_phi.json.gz',
    RSA_DIR  / 'deepalgpro_test_rsa.json.gz',
    RSA_DIR  / 'deepalgpro_test_disorder.json.gz',
    RSA_DIR  / 'deepalgpro_test_phi.json.gz',
    SS3_DIR  / 'iedb_positive_sequences_predictions.csv',
    CKPT_BASELINE,
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(f'  {p}' for p in missing))

print(f'RUN_TARGET:   {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'Results dir:  {RESULTS_DIR}')
print('All required files present.')

# Physics-Informed Feature Head: Binding Energy Constraints

## 0. Background and Motivation

### The gap between classification and mechanism

Allergenicity classifiers such as the frozen ESM-2 model in this project predict a binary label — allergenic or not — but offer limited insight into *why* a protein is an allergen. At the molecular level, allergenicity is driven by IgE-mediated responses that depend on CD4+ T-cell epitope presentation and subsequent antibody binding. Antibody–epitope binding free energy can be decomposed into physicochemical contributions:

$$\Delta G_{\text{bind}} = \Delta G_{\text{elec}} + \Delta G_{\text{vdW}} + \Delta G_{\text{solv}} + \Delta G_{\text{HB}} + \Delta G_{\text{entropy}}$$

Each term is governed by structural and sequence features that are at least partially accessible from per-residue predictions:

| ΔG term | Physical meaning | NetSurfP proxy used here |
|---------|-----------------|-------------------------|
| ΔG_elec | Electrostatic complementarity | charge, charge × RSA |
| ΔG_vdW | Shape complementarity / burial | hydrophobicity, (1−RSA) × hydrophobicity |
| ΔG_solv | Desolvation of binding interface | RSA |
| ΔG_HB | Hydrogen-bond networks at CDR contacts | HB donors + acceptors, HB × RSA |
| ΔG_entropy | Conformational flexibility penalty | disorder |

### Goal B: NetSurfP proxies, not computed energies

This experiment is **Goal B** — using NetSurfP-3.0 per-residue predictions (RSA, disorder, φ/ψ) as *proxies* for the ΔG terms rather than computed energy values. This distinction is important: true ΔG_bind terms require paired antibody–allergen complex structures and force-field calculations. No such paired structures exist for the DeepAlgPro dataset. Instead, we use computable per-residue descriptors that are known to correlate with binding energetics at the residue level.

### What this experiment tests

A `PhysicsProjection` layer — a single linear layer (10 → 1) — is added alongside the existing ESM-2 pooling stream. Its learned weights correspond to the effective contribution of each physicochemical channel to the allergenicity logit. The existing learned attention weights α are shared: they are computed once from ESM-2 residue embeddings and used to pool both the ESM embedding stream and the physics scalar stream. No second attention module is added.

The experiment addresses two limitations identified in the main paper:
1. **Interpretability**: learned `PhysicsProjection` weights can be inspected against known physical priors, making the model partially interpretable in ΔG terms.
2. **Mechanistic alignment**: if the model improves epitope alignment (attention AUROC on IEDB splitB) alongside classification performance, it provides evidence that the physics features capture epitope-relevant signal. If alignment stays flat or degrades, the misalignment finding from the main paper extends to physicochemical feature use.

## 1. Environment and Imports

Imports from the `xallergen.physics_head` package, seeds, and global plot styling. All outputs are written to `results/physics_head/` (Drive-backed on Colab).

In [ ]:
# Standard library
import json

# Numerical and data
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch

from xallergen.baseline_notebook_utils import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)

from xallergen.physics_head.features import PhysicsScaler, build_physics_vector
from xallergen.physics_head.model import (
    CHANNEL_NAMES, EXPECTED_SIGNS, PHYSICS_INIT_WEIGHTS,
    FrozenESM2WithPhysics, get_weight_summary,
)

# Global plot style — set once so all figures are consistent
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'font.family':     'sans-serif',
    'font.size':       11,
    'axes.titlesize':  12,
    'axes.labelsize':  11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

print(f'RESULTS_DIR: {RESULTS_DIR}')

## 2. Physics Feature Inspection

Before fitting the scaler or training the model, we inspect the raw physics features for a sample of five training proteins. Each of the 10 channels proxies a different ΔG term:

| Ch | Name | ΔG proxy | Physical rationale |
|----|------|----------|---------|
| 0 | RSA | ΔG_solv | Desolvation cost; surface availability for Ab binding |
| 1 | charge | ΔG_elec | Net formal charge at pH 7 |
| 2 | charge × RSA | ΔG_elec | Exposed charge is most relevant for CDR electrostatic contacts |
| 3 | hydrophobicity | ΔG_vdW | Kyte-Doolittle packing potential |
| 4 | (1−RSA) × hydrophob | ΔG_vdW | Burial-weighted hydrophobicity (hydrophobic core) |
| 5 | HB count | ΔG_HB | Total side-chain H-bond donors + acceptors |
| 6 | HB × RSA | ΔG_HB | Only exposed H-bond groups can engage CDR contacts |
| 7 | disorder | ΔG_entropy | Flexible regions impose an entropic binding penalty |
| 8 | sin(φ) | backbone | Encodes local secondary-structure geometry |
| 9 | cos(φ) | backbone | Complements ch 8 for full angular coverage |

In [ ]:
# Load training sequences and a sample of 5 physics feature vectors from the
# compact JSON files.  Print per-channel ranges to motivate normalisation.

from xallergen.physics_head.train import _load_physics_from_json
import gzip, json as _json

train_df = pd.read_csv(DATA_DIR / 'deepalgpro_train_cleaned.csv')
train_df['sequence_id'] = train_df['sequence_id'].astype(str)
train_df['sequence']    = train_df['sequence'].astype(str).str.strip().str.upper()

# Load physics features for all training sequences via compact JSON files
ns_train = _load_physics_from_json(
    RSA_DIR / 'deepalgpro_train_rsa.json.gz',
    RSA_DIR / 'deepalgpro_train_disorder.json.gz',
    RSA_DIR / 'deepalgpro_train_phi.json.gz',
    set(train_df['sequence_id']),
)
print(f'Physics features loaded for {len(ns_train)}/{len(train_df)} training sequences')

# Sample 5 proteins and build raw physics vectors
sample_df = (
    train_df[train_df['sequence_id'].isin(ns_train)]
    .sample(5, random_state=int(RANDOM_STATE))
    .reset_index(drop=True)
)
sample_vecs = {}
for row in sample_df.itertuples(index=False):
    ns = ns_train[row.sequence_id]
    sample_vecs[row.sequence_id] = build_physics_vector(
        row.sequence, ns['rsa'], ns['disorder'], ns['phi'], ns['psi'],
    )

# Print per-channel value ranges
all_raw = np.concatenate(list(sample_vecs.values()), axis=0)
print(f"\n{'Channel':<25} {'min':>8} {'max':>8} {'mean':>8} {'std':>8}")
print('-' * 63)
for i, name in enumerate(CHANNEL_NAMES):
    col = all_raw[:, i]
    print(f"{name:<25} {col.min():8.3f} {col.max():8.3f} {col.mean():8.3f} {col.std():8.3f}")

In [ ]:
# Heatmap of all 10 channels along the sequence of one example protein.
# Residues on the x-axis, channels on the y-axis.  Diverging colour scale
# (centred at 0) makes positive and negative values immediately distinguishable.

example_id  = sample_df.iloc[0]['sequence_id']
example_vec = sample_vecs[example_id]  # (L, 10)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    example_vec.T,
    ax=ax,
    cmap='RdBu_r',
    center=0,
    yticklabels=CHANNEL_NAMES,
    xticklabels=False,
    cbar_kws={'label': 'raw value', 'shrink': 0.7},
)
ax.set_xlabel(f'Residue position  (L = {example_vec.shape[0]})')
ax.set_title(f'Raw physics features — {example_id}')
fig.tight_layout()
plt.show()

## 3. Scaler Fitting

The table above shows that channels operate on very different scales — RSA is in [0, 1], Kyte-Doolittle hydrophobicity spans [−4.5, 4.5], and HB count reaches up to 5. Without normalisation, channels with large absolute values dominate the linear physics projection and distort gradient magnitudes.

We fit a per-channel zero-mean / unit-variance standardiser on the concatenation of all training residues. Channels 8 and 9 (sin φ, cos φ) are excluded: circular angles are already bounded in [−1, 1] and standardising them would distort the unit-circle interpretation of backbone dihedral geometry.

The scaler is fitted on training residues only to prevent data leakage, and saved to `results/physics_head/physics_scaler.json` so the same transform is applied identically at inference time.

In [ ]:
# Build the full (N_residues_total, 10) matrix from the already-loaded ns_train
# dict, fit the scaler, and plot the raw std per channel.

raw_vecs = []
for row in train_df.itertuples(index=False):
    sid = str(row.sequence_id)
    if sid not in ns_train:
        continue
    ns  = ns_train[sid]
    seq = str(row.sequence).upper()
    if len(seq) == len(ns['rsa']):
        raw_vecs.append(
            build_physics_vector(seq, ns['rsa'], ns['disorder'], ns['phi'], ns['psi'])
        )

all_residues = np.concatenate(raw_vecs, axis=0)  # (N_total, 10)
print(f'Total training residues: {all_residues.shape[0]:,}')

scaler = PhysicsScaler().fit(all_residues)

scaler_path = RESULTS_DIR / 'physics_scaler.json'
scaler.save(scaler_path)
print(f'Scaler saved to {scaler_path}')

print(f"\n{'Channel':<25} {'raw mean':>10} {'raw std':>10}")
print('-' * 47)
for i, name in enumerate(CHANNEL_NAMES[:8]):
    print(f"{name:<25} {scaler.mean[i]:10.3f} {scaler.std[i]:10.3f}")
print('(channels 8–9: sin/cos φ, excluded — already in [−1, 1])')

# Bar chart: raw std per channel (shows scale problem at a glance)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(range(8), scaler.std, color='#94a3b8', edgecolor='none')
ax.set_xticks(range(8))
ax.set_xticklabels(CHANNEL_NAMES[:8], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Raw standard deviation')
ax.set_title('Per-channel scale before normalisation (channels 0–7)')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

## 4. Model Initialisation

### Architectural change

`FrozenESM2WithPhysics` extends the baseline by adding a single physics stream:

1. The frozen ESM-2 backbone produces per-residue embeddings (B × L × embed_dim).
2. The learned attention module computes α (B × L) from those embeddings — **once**.
3. α pools the ESM embeddings → (B × embed_dim) and the physics scalar (output of `PhysicsProjection`) → (B × 1) simultaneously using the **same α**.
4. The two pooled vectors are concatenated → (B × embed_dim+1) → MLP → logit.

No second attention module is added. Sharing α keeps the trainable parameter count minimal and forces the model to allocate attention to positions that are jointly informative in both streams.

### Physics-informed initialisation

`PhysicsProjection` is initialised with `PHYSICS_INIT_WEIGHTS` (not random) so the model begins from a physically interpretable state and the optimiser starts close to a plausible solution.

In [ ]:
# Load the baseline frozen ESM-2 checkpoint (λ=0, no regularisation) and wrap
# it with FrozenESM2WithPhysics.  This gives the physics head a classification-
# optimised starting point while keeping the physics channel warm-started.

from xallergen.baseline_notebook_utils import load_baseline_checkpoint, detect_device

DEVICE = detect_device()
print(f'Device: {DEVICE}')

baseline_model, _ = load_baseline_checkpoint(CKPT_BASELINE, DEVICE)
model = FrozenESM2WithPhysics(baseline_model).to(DEVICE)

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParameter summary:')
print(f'  Total:     {n_total:>10,}')
print(f'  Trainable: {n_trainable:>10,}  (attention pool + physics proj + classifier)')
print(f'  Frozen:    {n_total - n_trainable:>10,}  (ESM-2 backbone)')

print('\nPhysicsProjection initial weights:')
rationale = {
    'rsa':                'surface-exposed → accessible to Ab',
    'charge':             'charged residues attract CDR complementary charge',
    'charge_x_rsa':       'exposed charge → direct electrostatic contact',
    'hydrophobicity':     'hydrophobic → prefers burial, not surface epitope',
    'hydrophob_x_burial': 'buried hydrophobic core → anti-epitope',
    'hb_count':           'H-bond capacity → stabilises CDR contacts',
    'hb_x_rsa':           'exposed H-bond groups → direct CDR engagement',
    'disorder':           'disordered → poor MHC-II binder (ΔG_entropy penalty)',
    'sin_phi':            'backbone geometry — no directional prior',
    'cos_phi':            'backbone geometry — no directional prior',
}
sign_str = {1: '+1', -1: '−1', 0: ' 0'}
print(f"  {'Channel':<25} {'weight':>7}  {'exp. sign':>10}  rationale")
print('  ' + '-' * 76)
for name, w, es in zip(CHANNEL_NAMES, PHYSICS_INIT_WEIGHTS.flatten().tolist(), EXPECTED_SIGNS):
    print(f"  {name:<25} {w:7.3f}  ({sign_str[es]:>2})         {rationale[name]}")

## 5. Training

Training uses the same hyperparameters as the baseline (notebook 03) — batch size 24, AdamW lr=1e-3 weight_decay=1e-4, up to 30 epochs, early stopping patience=5 min_delta=1e-3, 10 % validation split stratified by label, BCEWithLogitsLoss with class-imbalance pos_weight — so any performance difference is attributable to the physics head, not training configuration. If the checkpoint already exists the step is skipped.

In [ ]:
# Run training or load existing checkpoint/history.
# train.main() automatically prefers compact JSON files (Colab-compatible)
# over the full NetSurfP CSV.

from xallergen.physics_head import train as _train_module

ckpt_path    = RESULTS_DIR / 'checkpoints' / 'best.pt'
history_path = RESULTS_DIR / 'training_history.json'

if ckpt_path.exists():
    print(f'Checkpoint found at {ckpt_path} — skipping training.')
    history = json.loads(history_path.read_text())
else:
    history = _train_module.main(
        baseline_checkpoint_path=CKPT_BASELINE,
        rsa_json_path=RSA_DIR / 'deepalgpro_train_rsa.json.gz',
        disorder_json_path=RSA_DIR / 'deepalgpro_train_disorder.json.gz',
        phi_json_path=RSA_DIR / 'deepalgpro_train_phi.json.gz',
        output_dir=RESULTS_DIR,
        device=DEVICE,
    )

print(f'Epochs recorded: {len(history)}')

In [ ]:
# Plot training/validation loss and validation AUROC per epoch.
# Both panels together confirm convergence and identify the best checkpoint epoch.

epochs_run   = [r['epoch']      for r in history]
train_losses = [r['train_loss'] for r in history]
val_losses   = [r['val_loss']   for r in history]
val_aurocs   = [r['val_auroc']  for r in history]

best_epoch = history[int(np.argmin(val_losses))]['epoch']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(epochs_run, train_losses, color='#94a3b8', label='Train')
ax1.plot(epochs_run, val_losses,   color='#0f766e', label='Val')
ax1.axvline(best_epoch, color='k', linestyle=':', linewidth=0.9, alpha=0.6)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE loss'); ax1.set_title('Loss')
ax1.legend(frameon=False, fontsize=9)
ax1.spines[['top', 'right']].set_visible(False)

ax2.plot(epochs_run, val_aurocs, color='#0f766e')
ax2.axvline(best_epoch, color='k', linestyle=':', linewidth=0.9, alpha=0.6,
            label=f'Best epoch ({best_epoch})')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val AUROC'); ax2.set_title('Validation AUROC')
ax2.legend(frameon=False, fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

fig.suptitle('Training history — physics head', y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

## 6. Classification Performance

We evaluate the physics head and the baseline frozen ESM-2 on the held-out DeepAlgPro test set using identical batched inference.

Three possible outcomes:
- **Improves**: the explicit physicochemical channels encode information not captured by ESM-2 alone.
- **Flat**: the physics features are largely redundant with what ESM-2 encodes implicitly; the main contribution shifts to interpretability (section 8).
- **Degrades**: the physics head introduces noise or a conflicting inductive bias; likely caused by overly coarse NetSurfP proxies or the shared-α bottleneck.

In [ ]:
# Run evaluation (or load cached metrics) and evaluate the baseline on the
# same test set for direct comparison.  Both models use batched inference.

from xallergen.physics_head import evaluate as _eval_module
from xallergen.baseline_notebook_utils import build_tokenizer, HF_MODEL_NAME
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef, accuracy_score

metrics_path = RESULTS_DIR / 'metrics.json'
if metrics_path.exists():
    print(f'Metrics found at {metrics_path} — loading.')
    metrics = json.loads(metrics_path.read_text())
else:
    metrics = _eval_module.main(
        rsa_json_path=RSA_DIR / 'deepalgpro_test_rsa.json.gz',
        disorder_json_path=RSA_DIR / 'deepalgpro_test_disorder.json.gz',
        phi_json_path=RSA_DIR / 'deepalgpro_test_phi.json.gz',
        output_dir=RESULTS_DIR,
        device=DEVICE,
    )

ph_cls = metrics['classification']

# Batched baseline evaluation on the same test set
test_df = pd.read_csv(DATA_DIR / 'deepalgpro_test_cleaned.csv')
test_df['sequence_id'] = test_df['sequence_id'].astype(str)
test_df['sequence']    = test_df['sequence'].astype(str).str.strip().str.upper()
test_df['label']       = test_df['label'].astype(int)

tokenizer   = build_tokenizer(HF_MODEL_NAME)
EVAL_BATCH  = 32
all_logits, all_labels = [], []
baseline_model.eval()
for i in range(0, len(test_df), EVAL_BATCH):
    batch = test_df.iloc[i : i + EVAL_BATCH]
    enc   = tokenizer(
        batch['sequence'].tolist(),
        add_special_tokens=False, padding=True, truncation=False, return_tensors='pt',
    )
    with torch.no_grad():
        out = baseline_model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
    all_logits.extend(out['logits'].cpu().tolist())
    all_labels.extend(batch['label'].tolist())

bl_probs  = 1 / (1 + np.exp(-np.array(all_logits)))
bl_preds  = (bl_probs >= 0.5).astype(int)
bl_labels = np.array(all_labels)
bl_cls = {
    'auroc':    float(roc_auc_score(bl_labels, bl_probs)),
    'f1':       float(f1_score(bl_labels, bl_preds, zero_division=0)),
    'mcc':      float(matthews_corrcoef(bl_labels, bl_preds)),
    'accuracy': float(accuracy_score(bl_labels, bl_preds)),
}

comp = pd.DataFrame({
    'Metric':         ['AUROC', 'F1', 'MCC', 'Accuracy'],
    'Baseline ESM-2': [bl_cls['auroc'],  bl_cls['f1'],  bl_cls['mcc'],  bl_cls['accuracy']],
    'Physics Head':   [ph_cls['auroc'],  ph_cls['f1'],  ph_cls['mcc'],  ph_cls['accuracy']],
})
comp['Δ'] = comp['Physics Head'] - comp['Baseline ESM-2']
for col in ['Baseline ESM-2', 'Physics Head']:
    comp[col] = comp[col].map('{:.4f}'.format)
comp['Δ'] = comp['Δ'].map('{:+.4f}'.format)
print('Classification metrics on DeepAlgPro test set:')
print(comp.to_string(index=False))

## 7. Epitope Alignment

A key finding in the main paper is that the baseline ESM-2 attention weights do not align with IEDB CD4+ T-cell epitopes on the 61 splitB allergen proteins. We test whether the physics head changes this alignment by computing the per-protein AUROC of the model's attention weights α against the binary IEDB epitope mask.

Three possible outcomes:
- **Improves**: the physics features (surface accessibility, charge exposure, HB capacity) pull α toward epitope positions — supports the mechanistic hybrid model direction.
- **Flat**: the model exploits epitope-correlated physics channels but α does not shift toward epitopes, **strengthening** the misalignment finding.
- **Degrades**: the physics stream competes with whatever weak epitope signal exists in ESM-2 attention — the explicit channel is redundant or interfering.

In [ ]:
# Load per-protein epitope alignment results and display as a boxplot.
# The paired Wilcoxon p-value tests whether attention AUROC is significantly
# above the random baseline of 0.5 (equivalent to a one-sample test).

align       = metrics['epitope_alignment']
per_protein = pd.DataFrame(metrics['per_protein_alignment'])

p     = float(align['wilcoxon_p'])
stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

print(f"Epitope alignment on IEDB splitB ({align['n_proteins']} proteins):")
print(f"  Mean attention AUROC : {align['mean_attention_auroc']:.4f}")
print(f"  Wilcoxon p           : {p:.2e}  ({stars})")

auroc_vals = per_protein['auroc'].values
y_range    = auroc_vals.max() - auroc_vals.min()
annot_y    = auroc_vals.max() + y_range * 0.06

fig, ax = plt.subplots(figsize=(4, 5))
ax.boxplot(
    auroc_vals, widths=0.45, patch_artist=True,
    boxprops=dict(facecolor='#0f766e', alpha=0.8),
    medianprops=dict(color='white', linewidth=2),
    whiskerprops=dict(linewidth=1), capprops=dict(linewidth=1),
    flierprops=dict(marker='.', markersize=4, alpha=0.5),
)
ax.axhline(0.5, color='#475569', linestyle='--', linewidth=0.9, label='Random (0.5)')
ax.text(1, annot_y, f'{stars}\np = {p:.2e}', ha='center', va='bottom', fontsize=10)
ax.set_xticks([1]); ax.set_xticklabels(['Physics Head'])
ax.set_ylabel('Per-protein attention AUROC')
ax.set_title(f'Epitope alignment — splitB  (n = {align["n_proteins"]})')
ax.set_ylim(auroc_vals.min() - y_range * 0.05, annot_y + y_range * 0.18)
ax.legend(frameon=False, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

## 8. Learned Weight Analysis

The 10 scalar weights of `PhysicsProjection` can be read as a learned linear free-energy equation. We compare each learned weight against its physically motivated expected sign.

**Preserved sign**: the classifier uses the channel in a physically plausible direction (e.g., positive weight on RSA → surface-exposed residues predicted as more allergenic, consistent with antibody accessibility).

**Flipped sign**: the classifier exploits a channel in a direction inconsistent with known binding biophysics. Flipped signs extend the misalignment finding beyond residue positions to physicochemical feature use: the model may exploit biophysical features in statistically opportunistic but mechanistically implausible ways.

In [ ]:
# Load and display the weight summary table, then visualise as a bar chart.

from matplotlib.patches import Patch

weight_df = pd.read_csv(RESULTS_DIR / 'weight_summary.csv')

dg_term = {
    'rsa':                'ΔG_solv',
    'charge':             'ΔG_elec',
    'charge_x_rsa':       'ΔG_elec × access.',
    'hydrophobicity':     'ΔG_vdW',
    'hydrophob_x_burial': 'ΔG_vdW × burial',
    'hb_count':           'ΔG_HB',
    'hb_x_rsa':           'ΔG_HB × access.',
    'disorder':           'ΔG_entropy',
    'sin_phi':            'backbone',
    'cos_phi':            'backbone',
}
weight_df['dg_term'] = weight_df['channel_name'].map(dg_term)

disp = weight_df[['channel_name', 'dg_term', 'expected_sign', 'learned_weight', 'sign_preserved']].copy()
disp['learned_weight'] = disp['learned_weight'].map('{:.4f}'.format)
disp['sign_preserved'] = disp['sign_preserved'].map({True: 'yes', False: 'NO'})
print(disp.to_string(index=False))

n_directional = sum(1 for es in EXPECTED_SIGNS if es != 0)
n_preserved   = sum(
    1 for p, es in zip(weight_df['sign_preserved'], EXPECTED_SIGNS)
    if es != 0 and p
)
print(f'\nSign preserved for {n_preserved}/{n_directional} directional channels (sin/cos φ excluded from count)')

In [ ]:
# Horizontal bar chart of learned weights.
# Triangles mark the physically expected direction.
# Green = sign preserved; red = sign flipped.

w         = weight_df['learned_weight'].astype(float).to_numpy()
names     = weight_df['channel_name'].tolist()
expected  = weight_df['expected_sign'].astype(int).tolist()
preserved = weight_df['sign_preserved'].tolist()

bar_colors = ['#0f766e' if p else '#dc2626' for p in preserved]
marker_x   = max(float(np.abs(w).max()) * 0.18, 0.005)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.barh(range(10), w, color=bar_colors, alpha=0.85, edgecolor='none')
ax.axvline(0, color='k', linewidth=0.8)
for i, es in enumerate(expected):
    if es != 0:
        ax.plot(es * marker_x, i, f'k{"^" if es > 0 else "v"}', markersize=6, alpha=0.65, clip_on=False)
ax.set_yticks(range(10))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Learned weight')
ax.set_title('PhysicsProjection weights  (▲/▼ = expected sign)')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(
    handles=[Patch(color='#0f766e', label='Sign preserved'), Patch(color='#dc2626', label='Sign flipped')],
    frameon=False, fontsize=9,
)
fig.tight_layout()
plt.show()

## 9. Summary and Next Steps

### Classification result

If the physics head matches the baseline (section 6), ESM-2 already encodes most of the NetSurfP proxy information implicitly and the contribution of the physics head is interpretability rather than accuracy. Improvement indicates genuinely complementary signal. Degradation suggests the coarse Q3 proxies or the shared-α bottleneck introduce noise; remedies include using 8-class DSSP features or a separate (unfrozen) attention head for the physics stream.

### Epitope alignment result

If the physics head does not improve attention AUROC on IEDB splitB (section 7), the misalignment finding from the main paper is strengthened: even explicit guidance by epitope-correlated biophysical features does not shift attention toward epitope residues. The attention mechanism serves a discriminative role orthogonal to epitope selection. Improvement would support the mechanistic hybrid model direction.

### Weight sign analysis

Widespread sign preservation (section 8) indicates physically plausible learning. Flipped signs indicate the classifier exploits physicochemical features in statistically opportunistic but biophysically implausible directions — extending the misalignment narrative from residue positions to feature use, which strengthens the paper's argument for mechanistic hybrid models.

### Conditional next steps

- **Classification improves**: ablate each channel individually to identify the most informative ΔG proxy; report the physics head as a lightweight interpretable enhancement.
- **Classification flat + signs preserved**: compute Pearson r between per-residue physics weights and experimental ΔG decompositions for any allergen–antibody complexes in the PDB.
- **Signs flipped**: test whether flipped channels are correlated with the allergen label distribution; report as a data-distribution artefact and argue for paired Ab–allergen structures.
- **Alignment improves**: compare all model variants (notebooks 12–19) on IEDB splitB AUROC and position the physics head as the best-aligning variant.
- **Regardless**: ablate the shared-α design against a two-attention-module variant to isolate the contribution of α sharing from the physics features themselves.

In [ ]:
# Shut down the Colab runtime to free the GPU after all cells have run.
# Skip on local execution.
if IS_COLAB:
    import os
    os.kill(os.getpid(), 9)